In [ ]:
file_path = '/Volumes/workspace/default/sales/'
scheme_path = '/Volumes/workspace/default/sales/_schema'
checkpoint_path = '/Volumes/workspace/default/sales/_checkpoint'
df = spark.readStream.format('cloudFiles').option('cloudFiles.format','csv').option('cloudFiles.schemaLocation',scheme_path).option('cloudFiles.schemaEvolutionMode','rescue').option('cloudFiles.maxFilesPerTrigger',1).load(file_path)



In [ ]:

from pyspark.sql import functions as f

df = df.toDF(
'Transaction_ID',
 'Customer_ID',
 'Category',
 'Item',
 'Price_Per_Unit',
 'Quantity',
 'Total_Spent',
 'Payment_Method',
 'Location',
 'Transaction_Date',
 'Discount_Applied',
 'updated_time',
 '_rescued_data'
)

source_df = df.withColumn('source',f.col('_metadata.file_path')).withColumn('ingested_time',f.date_format(f.current_timestamp(),'yyyy-MM-dd HH:mm:ss'))
bronze_table = 'main.bronze.sales'
if spark.catalog.tableExists(bronze_table):
  bronze_df = spark.read.table(bronze_table)
  last_ingested_time = bronze_df.agg(f.max('updated_time').alias('max')).first()['max']
else:
    last_ingested_time = None

if last_ingested_time is None:
    incremental_df = source_df
else:
    incremental_df = source_df.filter(f.col('ingested_time') > last_ingested_time)



In [ ]:
query = incremental_df.writeStream.format('delta').outputMode('append').option('checkpointLocation',checkpoint_path).trigger(availableNow=True).toTable(bronze_table)
query.awaitTermination()